# ***IMPORTAR LIBRERIAS***

In [1]:
from datetime import datetime
import time

from selenium.webdriver import Chrome
from selenium import webdriver
#from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait as wait

import base64
import re
import csv

# ***FUNCIONES***

In [2]:
def actual_date():
    #no es necesario modificar el formato, coincide con el de Elux (yyyy-mm-dd)
    now = datetime.now()
    date_now = now.date()
    #print(date_now)
    return date_now

def extract_string(texto):
    # Definir la expresión regular
    patron = re.compile(r'\s*([^\/\s]*)\s*(?:\/\s*([^-\s]+(?:\s*-\s*[^\s]+)?))?')

    # Buscar coincidencias en el texto
    coincidencias = patron.search(texto)

    if coincidencias:
        # Obtener las cadenas de las coincidencias
        primera_cadena = coincidencias.group(1)
        segunda_cadena = coincidencias.group(2) or ""  # Usar cadena vacía si no hay coincidencia en el segundo grupo

        return primera_cadena, segunda_cadena
    else:
        return None, None
    
def otm_to_mturn(list_element):
    date = actual_date()
    name_file = "OTM_TO_MTURN"+str(date)+".csv"
    headers = "Hora,Tractora,Remolque\n"
    
    with open(name_file, mode='w') as file:
        
        file.write(headers)

        for line in list_element:
            file.write(line+"\n")
    

# ***CONFIGURAR NAVEGADOR***

In [3]:
# ****** URL DE PRODUCCION ******
url = "https://www.transportation.electrolux.com/OTM-ADF/faces/glog/fusion/core/view/OTMHome.jsf"
 
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

driver_options = Options()
driver_options.add_argument(headers)
driver_options.add_argument('--no-sandbox')

driver = webdriver.Chrome()
driver.maximize_window()
driver.get(url)
#driver.execute_script("window.scrollTo(500, 1080);")

time.sleep(5)

# ***ENTRAR EN LA PAGINA "APPOINTMENT FOR GATE"***

In [4]:
#loguear

decoded_user = base64.b64decode("R09OWkFKT1MwMQ==")
decoded_pw = base64.b64decode("RWxlY3Ryb2x1eDIwMjY=")

user = decoded_user.decode("utf-8")
passw = decoded_pw.decode("utf-8")

user_login = driver.find_element("xpath", "/html/body/table/tbody/tr[3]/td/table/tbody/tr/td/form/table/tbody/tr[2]/td/input").send_keys(user)
user_passw = driver.find_element("xpath", "/html/body/table/tbody/tr[3]/td/table/tbody/tr/td/form/table/tbody/tr[3]/td/input").send_keys(passw)
user_passw = driver.find_element("xpath", "/html/body/table/tbody/tr[3]/td/table/tbody/tr/td/form/table/tbody/tr[3]/td/input").send_keys(Keys.RETURN)

time.sleep(3)

#acceder al menú correspondiente

driver.find_element("xpath", "/html/body/div[2]/form/div[1]/div[2]/div/div[2]/div/div[1]/div/div/div[1]/div[1]/div[1]/a").click()
time.sleep(5)

# ***CAMBIAMOS DE IFRAME***

In [5]:

wait(driver, 3).until(EC.frame_to_be_available_and_switch_to_it("mcr:1:if1::f"))


True

# ***INTRODUCIR DATOS PARA VER LA PARRILLA***

In [6]:
#Metemos fecha y definimos plataforma
date = str(actual_date())

plat = "ES51-SAP"

try:
    send_plat = driver.find_element("xpath", '/html/body/form/div[1]/div/div[1]/table/tbody/tr[5]/td/div/table/tbody/tr/td[1]/input[1]').send_keys(plat)
    driver.find_element("xpath", "/html/body/form/div[1]/div/div[1]/table/tbody/tr[1]/td/div/input[1]").send_keys(date)
except Exception as e:
    print(e)

driver.find_element("xpath", "/html/body/form/div[2]/table/tbody/tr/td[1]/div/a").click()
time.sleep(7)

# ***VOLVEMOS A CAMBIAR DE IFRAME***

In [7]:
wait(driver, 3).until(EC.frame_to_be_available_and_switch_to_it("reportFrame"))

True

# ***SACAMOS REGISTROS***

In [8]:
#Primera posicion de la parrilla
count_hour = 2
count_plate =2
separator = ","
list_element = []

for  i in range(1, 70):

    try:
        plate = driver.find_element("xpath", "/html/body/table/tbody/tr[2]/td/table[2]/tbody/tr["+str(count_plate)+"]/td[22]/div").text
        slot = driver.find_element("xpath","/html/body/table/tbody/tr[2]/td/table[2]/tbody/tr["+str(count_hour)+"]/td[5]/div" ).text

        #Dividimos la cadena plate
        tract, remol = extract_string(plate)
        
        
        if slot == "" :
            break
        else:
            result = slot + separator + str(tract) + separator + remol
            list_element.append(result)
            
            
            #print(list_element)
            count_hour = count_hour+1
            count_plate = count_plate+1
    except:
        break
otm_to_mturn(list_element)
for i in list_element:
    print(i)



07:00,1010-KBF,
07:00,2371-FHF,
07:00,4791-JKV,R-3779-BDG
08:00,0897-FYL,
10:00,7521-KHH,R-8880-BBZ
10:00,R-5043-BBT,3950-CNB
11:00,7115-KMP,R-9727-BCY
11:00,7860-MKL,R-0024-BCN
12:00,3278-LWT,R-0154-BDC
12:00,21-XP-89,L-209294
13:00,8980-KSV,R-2815-BCP
13:00,5820-KZZ,R-6097-BCT
13:00,BD62DC,LE3444
14:00,4791-JKV,R-7585-BCK
18:00,,
18:00,,
18:00,,


# ***CERRAR VENTANA***

In [9]:
#driver.close()
#print("Close conection")